# Encontrar impulsores de demanda con PROC GLMSELECT: paso a paso, LASSO y selección hacia adelante validada

## Resumen ejecutivo

Un equipo de analítica minorista usa **PROC GLMSELECT** para descubrir qué palancas de marketing y precios realmente mueven las ventas semanales de unidades, separando los verdaderos impulsores de demanda del ruido. La selección paso a paso puntuada por SBC, LASSO con validación cruzada de 5 particiones, y una búsqueda hacia adelante validada con datos de reserva recuperan todas el **mismo conjunto de verdaderos impulsores** — precio propio, precio de la competencia, gasto publicitario, volumen de correos electrónicos, promociones, feriados, la región Noreste y el canal Online — y cada uno de los cuatro señuelos plantados (`temp_f`, `noise1`, `noise2`, `noise3`) se descarta automáticamente.

Los métodos también coinciden estrechamente en las magnitudes: cada uno estima un efecto de precio propio cercano a **-28 unidades por dólar** y un efecto de precio de la competencia cercano a **+14**, los signos de sustitución que la ecuación generadora de datos incorporó. El único desacuerdo está en el margen — el LASSO validado por cruzada además conserva un pequeño contraste `Region=Midwest` estadísticamente insignificante (estimación -8.3 con un error estándar de 8.3) que tanto la selección paso a paso como la hacia adelante descartan. Una lista de impulsores que sobrevive a tres filosofías de selección diferentes es mucho más confiable que una ajustada a una sola regla.

## Fuentes de datos

Todos los datos en este cuaderno son **sintéticos** y se generan en línea (sin archivos externos, semilla `20260531`). Imitan una temporada de paneles de demanda tienda-semana para un minorista de bienes de consumo.

| Conjunto de datos | Filas | Grano | Variables clave |
|---------|------|-------|---------------|
| `demand` | 100 | Tienda-semana | `units` (respuesta: unidades semanales vendidas); `price`, `competitor` (precio propio y de la rival en anaquel); `adspend`, `email` (presión mediática); `promo`, `holiday` (banderas de evento); `region`, `channel` (efectos CLASS); `temp_f`, `noise1`-`noise3` (predictores señuelo/irrelevantes) |

`units` se construye a partir de una ecuación lineal conocida para que el conjunto "correcto" de impulsores sea verificable; `temp_f` y las tres columnas `noise` no llevan señal verdadera y existen para probar si cada método de selección las descarta.

# Encontrar impulsores de demanda con PROC GLMSELECT

Un gerente de categoría tiene docenas de explicativas candidatas para las ventas semanales: precio propio, precio de la competencia, gasto publicitario, volumen de correos electrónicos, promociones, feriados, región de la tienda, canal de venta, incluso el clima. Meterlas todas en una sola regresión invita al sobreajuste y a coeficientes inestables. **PROC GLMSELECT** automatiza la búsqueda de un modelo parsimonioso, con soporte para selección paso a paso, LASSO, elastic net y selección de ángulo mínimo, con validación cruzada y partición de reserva incorporadas.

En este cuaderno:

1. Generamos un panel de demanda tienda-semana sintético realista con un conjunto *conocido* de verdaderos impulsores (más variables señuelo deliberadas).
2. Ejecutamos **selección paso a paso** puntuada por SBC.
3. Ejecutamos **LASSO** con validación cruzada de 5 particiones.
4. Ejecutamos **selección hacia adelante validada con una reserva del 30%**.

Un buen método de selección debería recuperar los impulsores reales y descartar el ruido — veamos.

## 1. Generar un panel de demanda sintético

La respuesta `units` se construye a partir de una ecuación lineal explícita, así que conocemos la verdad fundamental: el precio y el precio de la competencia, el gasto publicitario, el volumen de correos electrónicos, las banderas de promoción y feriado, más los efectos de región y canal importan todos. Las variables `temp_f`, `noise1`, `noise2` y `noise3` son señuelos puros sin relación real con las ventas. Una semilla `call streaminit` hace que los datos sean reproducibles.

In [1]:
/* ---------------------------------------------------------------
   Panel de demanda tienda-semana sintético para un minorista de bienes de consumo.
   'units' sigue una ecuación CONOCIDA; temp_f y noise1-3 son señuelos.
   --------------------------------------------------------------- */
DATOS demand;
    LLAMAR streaminit(20260531);
    LONGITUD region $9 channel $8 promo $3;
    HACER store_week = 1 HASTA 100;
        /* Mezcla de región */
        u1 = rand('uniform');
        SI u1 < 0.34 ENTONCES region = 'Northeast';
        SINO SI u1 < 0.67 ENTONCES region = 'Midwest';
        SINO region = 'South';

        /* Canal de venta */
        SI rand('uniform') < 0.45 ENTONCES channel = 'Store';
        SINO channel = 'Online';

        /* Precios y palancas de medios */
        price      = round(8  + rand('uniform') * 12, 0.01);
        competitor = round(8  + rand('uniform') * 12, 0.01);
        adspend    = round(rand('gamma', 2) * 500, 1);
        email      = round(rand('uniform') * 100, 1);

        /* Banderas de evento y un señuelo climático irrelevante */
        temp_f     = round(40 + rand('uniform') * 50, 0.1);
        holiday    = (rand('uniform') < 0.12);
        SI rand('uniform') < 0.40 ENTONCES promo = 'Yes';
        SINO promo = 'No';

        /* Predictores de ruido puro (sin efecto verdadero) */
        noise1 = rand('normal');
        noise2 = rand('normal');
        noise3 = rand('normal');

        /* Unidades semanales vendidas a partir de una ecuación estructural conocida */
        units = 900
              - 28   * price
              + 14   * competitor
              + 0.06 * adspend
              + 1.2  * email
              + 55   * (promo = 'Yes')
              + 70   * holiday
              + 40   * (region = 'Northeast')
              + 25   * (channel = 'Online')
              + 30   * rand('normal');
        SI units < 0 ENTONCES units = 0;
        SALIDA;
    END;
    MANTENER region channel promo price competitor adspend email temp_f
         holiday noise1 noise2 noise3 units;
EJECUTAR;


NOTE: DATA demand


NOTE: Wrote demand (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds


## 2. Perfilar los datos

Antes de modelar, confirmar que la respuesta y las principales candidatas continuas están en escalas razonables.

In [2]:
PROCEDIMIENTO MEDIAS DATOS=demand n mean std min max maxdec=1;
    VAR units price competitor adspend email;
    ETIQUETA units="Unidades vendidas (semanales)" price="Precio propio"
          competitor="Precio de la competencia" adspend="Gasto publicitario"
          email="Volumen de correos electrónicos";
    TÍTULO "Demanda semanal y factores candidatos";
EJECUTAR;

                                         Demanda semanal y factores candidatos                                          

                                                  The MEANS Procedure

 Variable    Label                                    N        Mean     Std Dev     Minimum     Maximum
 ------------------------------------------------------------------------------------------------------
 units       Unidades vendidas (semanales)          100       874.8       136.3       508.6      1162.3
 price       Precio propio                          100        14.0         3.4         8.0        19.9
 competitor  Precio de la competencia               100        13.8         3.4         8.1        19.9
 adspend     Gasto publicitario                     100       990.6       726.9        50.0      3358.0
 email       Volumen de correos electrónicos        100        45.5        26.4         0.0        99.0
 ------------------------------------------------------------------------------


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 3. Selección paso a paso puntuada por SBC

La selección paso a paso alternadamente agrega y elimina efectos, aquí juzgada por el **Criterio de información bayesiano de Schwarz (SBC)** tanto para la prueba de entrada (`select=sbc`) como para la elección final del modelo (`choose=sbc`). SBC penaliza la complejidad más fuertemente que AIC, favoreciendo modelos más esbeltos.

Instrucciones y opciones clave:

- `CLASS region channel promo / param=reference` declara los predictores categóricos con codificación de celda de referencia.
- `selection=stepwise(select=sbc choose=sbc)` impulsa la búsqueda.
- `details=summary` imprime el resumen de selección paso a paso; `stb` agrega coeficientes estandarizados para que los efectos en distintas escalas sean comparables.
- `output out=demand_scored p=predicted r=residual` guarda los valores ajustados y los residuos para el puntaje posterior.

Lea el **Resumen de selección paso a paso** como el rastro de la búsqueda: comienza desde el modelo completo de 12 efectos y *elimina* efectos uno a la vez, descartando `noise1`, `noise2`, `temp_f`, el contraste `Region=Midwest` y `noise3` en su turno, porque cada eliminación reduce el SBC. Lo que sobrevive en la tabla de **Estimaciones de parámetros** es el modelo elegido.

In [3]:
PROCEDIMIENTO glmselect DATOS=demand seed=20260531;
    CLASE region channel promo / PARAM=reference;
    MODELO units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=stepwise(select=sbc choose=sbc)
          details=summary stb;
    ETIQUETA units="Unidades vendidas (semanales)" region="Región" channel="Canal"
          promo="Promoción" price="Precio propio" competitor="Precio de la competencia"
          adspend="Gasto publicitario" email="Volumen de correos electrónicos"
          temp_f="Temperatura (°F)" holiday="Feriado"
          noise1="Ruido 1" noise2="Ruido 2" noise3="Ruido 3";
    SALIDA out=demand_scored p=predicted r=residual;
    TÍTULO "Selección paso a paso de los impulsores de demanda (SBC)";
EJECUTAR;

                                         Demanda semanal y factores candidatos                                          

The GLMSELECT Procedure


Dependent Variable: UNITS Unidades vendidas (semanales)


Number of Observations Used: 100

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                  Stepwise Selection Summary                                                  

    Step    Action             Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)
--------  --------  -----------------  -----------------  --------  --------  --------  --------  --------  --------  --------
       1   Removed            Ruido 1                 12    0.9507    0.9439  707.0094  711.2420  713.1843  740.


NOTE: PROC GLMSELECT data=demand

NOTE: The data set WORK.DEMAND_SCORED has 100 observations and 15 variables.
NOTE: PROC GLMSELECT statement used.


## 4. LASSO con validación cruzada

El LASSO reduce los coeficientes hacia cero, realizando selección y regularización simultáneamente. En lugar de detenerse en un criterio fijo, dejamos que la **validación cruzada de 5 particiones** elija el punto en la trayectoria del LASSO con el mejor error fuera de muestra.

- `selection=lasso(choose=cv stop=none)` traza la trayectoria completa del LASSO y selecciona el paso óptimo por VC.
- `cvmethod=random(5)` asigna las observaciones a 5 particiones aleatorias.

El **Resumen de selección LASSO** muestra el orden en que los efectos entran a medida que la penalización se relaja: `price` primero, luego `adspend`, `competitor`, la región Noreste, `email`, `promo` y `holiday` — las siete señales verdaderas entran antes que cualquier señuelo. La validación cruzada luego elige el paso donde el PRESS fuera de muestra es más bajo, y la tabla de **Estimaciones de parámetros** resultante conserva exactamente los impulsores genuinos (más un pequeño término `Region=Midwest`) mientras excluye `temp_f`, `noise1`, `noise2` y `noise3`, que solo entran al final mismo de la trayectoria.

In [4]:
PROCEDIMIENTO glmselect DATOS=demand seed=20260531;
    CLASE region channel promo / PARAM=reference;
    MODELO units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=lasso(choose=cv stop=none)
          cvmethod=random(5);
    ETIQUETA units="Unidades vendidas (semanales)" region="Región" channel="Canal"
          promo="Promoción" price="Precio propio" competitor="Precio de la competencia"
          adspend="Gasto publicitario" email="Volumen de correos electrónicos"
          temp_f="Temperatura (°F)" holiday="Feriado"
          noise1="Ruido 1" noise2="Ruido 2" noise3="Ruido 3";
    TÍTULO "LASSO con validación cruzada de 5 particiones";
EJECUTAR;

                                         Demanda semanal y factores candidatos                                          

The GLMSELECT Procedure


Dependent Variable: UNITS Unidades vendidas (semanales)


Number of Observations Used: 100

  Cross Validation Information   

                   Item     Value
-----------------------  --------
Cross Validation Method    Random
        Number of Folds         5

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                                 LASSO Selection Summary                                                                  

    Step    Action                            Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  -


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 5. Selección hacia adelante validada con una reserva

Una tercera estrategia, complementaria: construir el modelo por **selección hacia adelante** (los efectos solo entran, nunca salen), pero detenerse donde el desempeño en una muestra de reserva independiente es mejor — protegiendo directamente contra el sobreajuste.

- `partition fraction(validate=0.30)` reserva aleatoriamente el 30% de las filas para validación, dejando 70 observaciones de entrenamiento y 30 de validación.
- `selection=forward(select=aic choose=validate)` agrega efectos por AIC pero elige el modelo final por el error de la muestra de validación.

La tabla de **Fracciones de partición** confirma la división 70/30. La selección hacia adelante luego incorpora efectos hasta que el error de validación deja de mejorar; los ocho efectos en la tabla final de **Estimaciones de parámetros** son precisamente los verdaderos impulsores, sin que los cuatro señuelos sean admitidos nunca. Cuando tres métodos construidos sobre principios distintos llegan a los mismos impulsores, es mucho más probable que el modelo sea real y no un artefacto de una sola regla de selección.

In [5]:
PROCEDIMIENTO glmselect DATOS=demand seed=20260531;
    CLASE region channel promo / PARAM=reference;
    MODELO units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=forward(select=aic choose=validate);
    partition fraction(validate=0.30);
    ETIQUETA units="Unidades vendidas (semanales)" region="Región" channel="Canal"
          promo="Promoción" price="Precio propio" competitor="Precio de la competencia"
          adspend="Gasto publicitario" email="Volumen de correos electrónicos"
          temp_f="Temperatura (°F)" holiday="Feriado"
          noise1="Ruido 1" noise2="Ruido 2" noise3="Ruido 3";
    TÍTULO "Selección hacia adelante validada con una reserva del 30%";
EJECUTAR;

                                         Demanda semanal y factores candidatos                                          

The GLMSELECT Procedure


Dependent Variable: UNITS Unidades vendidas (semanales)


Number of Observations Used: 70               
Number of Observations Used for Validation: 30

Partition Fractions 

      Role  Fraction
----------  --------
  Training    0.7000
Validation    0.3000
   Testing    0.0000

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                                Forward Selection Summary                                                                 

    Step    Action                            Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 6. Interpretación de los resultados

Las tres estrategias de selección recuperan el **mismo conjunto de verdaderos impulsores de demanda** y descartan cada señuelo. Leyendo directamente de las tablas de **Estimaciones de parámetros**:

- **El precio** es el impulsor dominante. Su coeficiente estandarizado en el modelo paso a paso es **-0.70**, por mucho el mayor en magnitud, y la pendiente en bruto se ubica entre **-27.8** (paso a paso y LASSO) y **-28.6** (hacia adelante) unidades por dólar — casi exactamente el -28 con el que se generaron los datos. **El precio de la competencia** mueve la demanda en la dirección opuesta en aproximadamente **+14.4** en los tres ajustes, el comportamiento de sustitución que un gerente de categoría espera.
- **El gasto publicitario** (alrededor de +0.062 unidades por dólar) y el **volumen de correos electrónicos** (alrededor de +1.2 unidades por envío) elevan ambos las ventas, cuantificando la respuesta a los medios.
- **Las promociones** y los **feriados** llevan los mayores efectos de evento: el contraste `Promo=No` se ubica alrededor de **-51 a -57** relativo a una semana promocionada, y el aumento por feriado es de aproximadamente **+66 a +76** unidades. La **región Noreste** (alrededor de +49 a +55) y el **canal Online** (alrededor de +24 a +32) llevan efectos de línea base positivos.
- Fundamentalmente, cada señuelo — `temp_f`, `noise1`, `noise2`, `noise3` — es **descartado** por la selección paso a paso y hacia adelante, y queda excluido del modelo LASSO elegido por VC. Cada método recuperó la señal estructural e ignoró el ruido.

Los tres métodos no son idénticos byte a byte: la selección paso a paso (SBC) y la búsqueda hacia adelante validada con reserva se asientan en los mismos ocho efectos, mientras que el LASSO validado por cruzada además retiene un pequeño contraste `Region=Midwest` (-8.3, error estándar 8.3) — una diferencia en el piso de ruido más que un desacuerdo sustantivo. Que una lista de impulsores sobreviva a SBC paso a paso, LASSO validado por cruzada y selección hacia adelante validada con reserva es la verdadera conclusión: un hallazgo robusto a tres filosofías de selección distintas es mucho más creíble que uno ajustado a un solo criterio. Con `OUTPUT OUT=demand_scored` capturando los valores ajustados y los residuos, el mismo flujo de trabajo se extiende naturalmente al puntaje del calendario de precios y promociones planeado para el próximo trimestre.